In [0]:
# Step 1: Load tweets.avro into a Spark DataFrame and inspect its schema
#
# Note: Unlike CSV or Parquet, Avro requires .format("avro").load(...)
# Also notice in printSchema() that 'user' is a nested struct containing
# user.username, user.display_name, and user.country!

catalog = "workspace"
schema = "retail_lab"
avro_path = f"/Volumes/{catalog}/{schema}/raw_files/tweets.avro"

tweets_df = spark.read.format("avro").load(avro_path)

tweets_df.printSchema()
display(tweets_df.limit(5))

root
 |-- tweet_id: long (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- text: string (nullable = true)
 |-- lang: string (nullable = true)
 |-- source: string (nullable = true)
 |-- is_retweet: boolean (nullable = true)
 |-- reply_to_tweet_id: long (nullable = true)
 |-- user: struct (nullable = true)
 |    |-- user_id: long (nullable = true)
 |    |-- username: string (nullable = true)
 |    |-- display_name: string (nullable = true)
 |    |-- verified: boolean (nullable = true)
 |    |-- followers_count: integer (nullable = true)
 |    |-- following_count: integer (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- account_created: timestamp (nullable = true)
 |-- hashtags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- mentions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- metrics: struct (nullable = true)
 |    |-- likes: integer (nullable = true)
 |    |-- retweets: integer (null

tweet_id,created_at,text,lang,source,is_retweet,reply_to_tweet_id,user,hashtags,mentions,metrics,geo
9000000000,2026-10-12T20:17:15.000Z,RT Thread on why #Travel #Music #Movies matters more than ever @aarav_muller101,en,Twitter Web App,true,null,"List(1001, ingrid_gupta1, Ingrid Gupta, false, 105, 75, US, 2026-03-24T00:00:00.000Z)","List(Travel, Music, Movies)",List(aarav_muller101),"List(0, 0, 0, 0)","List(Austin, 30.23235887367635, -97.75314143413839)"
9000000001,2026-10-28T22:20:49.000Z,Thread on why this matters more than ever,en,Twitter Web App,false,null,"List(1001, ingrid_gupta1, Ingrid Gupta, false, 105, 75, US, 2026-03-24T00:00:00.000Z)",List(),List(),"List(2, 0, 0, 0)",null
9000000002,2026-10-05T18:37:49.000Z,Anyone else following this today?,fr,Twitter for Android,false,null,"List(1197, emma_tanaka197, Emma Tanaka, false, 56, 1114, FR, 2016-09-25T00:00:00.000Z)",List(),List(),"List(3, 0, 0, 0)",null
9000000003,2026-10-14T09:26:25.000Z,Morning coffee and some this reading @rohan_kim55,en,TweetDeck,false,null,"List(1161, mei_singh161, Mei Singh, false, 164, 426, NG, 2023-07-27T00:00:00.000Z)",List(),List(rohan_kim55),"List(3, 1, 0, 0)","List(Lagos, 6.524446586148214, 3.4277842529452047)"
9000000004,2026-10-25T21:17:17.000Z,Just tried this and I'm impressed! @ananya_gupta5,en,Twitter for Android,false,null,"List(1000, aarav_khan0, Aarav Khan, false, 63, 295, US, 2018-04-21T00:00:00.000Z)",List(),List(ananya_gupta5),"List(1, 0, 0, 0)",null


In [0]:
# Question 1: Identify the top 5 users with the highest number of tweets
#
# Explanation:
# 1. Because 'user' is a nested structure (struct) inside the Avro file, we access
#    the username using dot notation: "user.username" (and "user.display_name").
# 2. .groupBy("user.username", "user.display_name").agg(F.count("*").alias("tweet_count")):
#    Counts how many tweets each user has posted.
# 3. .orderBy(F.desc("tweet_count")).limit(5): Sorts from highest to lowest and keeps the top 5.

from pyspark.sql import functions as F

top_5_users = (
    tweets_df.groupBy(
        F.col("user.username").alias("username"),
        F.col("user.display_name").alias("display_name")
    )
    .agg(F.count("*").alias("tweet_count"))
    .orderBy(F.desc("tweet_count"))
    .limit(5)
)

display(top_5_users)

username,display_name,tweet_count
aarav_khan0,Aarav Khan,304
ingrid_gupta1,Ingrid Gupta,187
noah_sato2,Noah Sato,134
ingrid_kumar3,Ingrid Kumar,111
emma_kumar4,Emma Kumar,70


In [0]:
# Question 2: Most commonly used words in all the tweets, excluding stopwords
#
# Explanation:
# 1. stopwords: A list of common filler words (e.g., "the", "and", "to", "is", "in", etc.) to exclude.
# 2. F.lower(F.col("text")): Converts all tweets to lowercase so "Data" and "data" count as the same word.
# 3. F.regexp_replace(..., r"[^a-z\s]", " "): Removes punctuation, numbers, and symbols.
# 4. F.split(..., r"\s+"): Splits each tweet sentence into a list (array) of individual words.
# 5. F.explode(...): Turns each word in the list into its own separate row so we can count them.
# 6. .filter(...): Removes empty strings, single-letter noise, and any word in our stopwords list.
# 7. .groupBy("word").agg(F.count("*")): Counts how many times each word appears across all tweets.

stopwords = [
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for", "of", "with", "by",
    "from", "up", "about", "into", "over", "after", "is", "am", "are", "was", "were", "be",
    "been", "being", "have", "has", "had", "do", "does", "did", "will", "would", "shall",
    "should", "can", "could", "may", "might", "must", "it", "its", "this", "that", "these",
    "those", "i", "you", "he", "she", "we", "they", "me", "him", "her", "us", "them", "my",
    "your", "his", "our", "their", "as", "if", "so", "than", "too", "very", "just", "not",
    "no", "yes", "rt", "http", "https", "co", "amp",
    # Common Spanish stopwords as well (since some tweets are in Spanish):
    "el", "la", "los", "las", "un", "una", "unos", "unas", "y", "o", "pero", "de", "del",
    "al", "en", "con", "por", "para", "sin", "sobre", "es", "son", "fue", "era", "ser",
    "estar", "esta", "este", "estos", "estas", "que", "se", "su", "sus", "mi", "mis", "tu",
    "tus", "lo", "le", "les", "nos", "como", "mas", "más", "ya", "muy", "si", "sí"
]

words_df = (
    tweets_df
    .select(
        F.explode(
            F.split(
                F.regexp_replace(F.lower(F.col("text")), r"[^a-záéíóúñü\s]", " "),
                r"\s+"
            )
        ).alias("word")
    )
    .filter((F.length(F.col("word")) > 1) & (~F.col("word").isin(stopwords)))
    .groupBy("word")
    .agg(F.count("*").alias("word_count"))
    .orderBy(F.desc("word_count"))
)

display(words_df.limit(20))

word,word_count
overrated,403
hot,403
take,403
stop,399
thinking,399
today,390
anyone,390
else,390
following,390
some,385


In [0]:
# Question 3: Filter the tweets that are in Spanish
#
# Explanation:
# The Avro dataset has a 'lang' column indicating the language of each tweet
# (using the ISO code 'es' for Español / Spanish).
# We filter where lang is 'es' (or 'spanish') and display the matching tweets.

spanish_tweets = tweets_df.filter(F.lower(F.col("lang")).isin("es", "spanish"))

print(f"Total tweets in Spanish: {spanish_tweets.count()}")
display(
    spanish_tweets.select(
        "tweet_id",
        "created_at",
        F.col("user.username").alias("username"),
        "lang",
        "text"
    )
)

Total tweets in Spanish: 0


tweet_id,created_at,username,lang,text


In [0]:
# Question 4: Which country has the highest number of tweets?
#
# Explanation:
# 1. 'country' is stored inside the nested 'user' struct as "user.country".
# 2. We group by "user.country", count the tweets per country, and sort from highest to lowest.
#    The first row (#1) is the country with the highest number of tweets!

tweets_by_country = (
    tweets_df.groupBy(F.col("user.country").alias("country"))
    .agg(F.count("*").alias("tweet_count"))
    .orderBy(F.desc("tweet_count"))
)

display(tweets_by_country)

country,tweet_count
US,859
CA,362
JP,333
DE,274
IN,259
AU,256
NG,221
BR,196
FR,135
GB,105


In [0]:
# Question 5: Separate the tweets by a specific username (e.g., all tweets by one specific user)
#
# Explanation:
# 1. top_5_users.first()["username"]: Grabs the top username from Question 1 into target_username.
# 2. .filter(F.col("user.username") == target_username): Keeps only the tweets posted by that user.
# 3. .select(F.col("user.username").alias("username"), F.col("user.display_name").alias("display_name"),
#    F.col("text").alias("tweet"), F.col("created_at").alias("timestamp")):
#    Selects and renames the username, display_name, tweet text, and timestamp columns.
# 4. .orderBy(F.desc("timestamp")): Sorts the user's tweets from newest to oldest.

target_username = top_5_users.first()["username"]
print(f"Filtering all tweets by username: {target_username}")

user_tweets = (
    tweets_df
    .filter(F.col("user.username") == target_username)
    .select(
        F.col("user.username").alias("username"),
        F.col("user.display_name").alias("display_name"),
        F.col("text").alias("tweet"),
        F.col("created_at").alias("timestamp")
    )
    .orderBy(F.desc("timestamp"))
)

display(user_tweets)

Filtering all tweets by username: aarav_khan0


username,display_name,tweet,timestamp
aarav_khan0,Aarav Khan,Thread on why #PySpark matters more than ever,2026-10-31T21:40:46.000Z
aarav_khan0,Aarav Khan,Anyone else following #MachineLearning #DataEngineering today? @ananya_kim132 @hiro_nielsen199,2026-10-31T21:22:37.000Z
aarav_khan0,Aarav Khan,Thread on why this matters more than ever @chen_khan186 @noah_kim35,2026-10-31T19:59:02.000Z
aarav_khan0,Aarav Khan,Can't stop thinking about #Python #BigData,2026-10-31T19:02:35.000Z
aarav_khan0,Aarav Khan,Can't stop thinking about this,2026-10-31T06:05:49.000Z
aarav_khan0,Aarav Khan,"RT Learning #Music this weekend, wish me luck",2026-10-31T01:02:19.000Z
aarav_khan0,Aarav Khan,Big news around this right now @ingrid_johnson81,2026-10-31T00:53:11.000Z
aarav_khan0,Aarav Khan,Anyone else following #PySpark today? @fatima_gupta178,2026-10-30T18:58:32.000Z
aarav_khan0,Aarav Khan,Thread on why #MachineLearning matters more than ever,2026-10-30T18:19:51.000Z
aarav_khan0,Aarav Khan,Big news around #BigData #AI right now @olivia_haddad193 @sofia_gupta86,2026-10-30T16:08:27.000Z
